# Notebook 02 — Runtime Landscape and Deployment Tiers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/02_runtime_landscape_and_deployment_tiers.ipynb)

This notebook maps the open-model runtime landscape for 2026. We compare the raw `transformers` baseline, `llama.cpp`, `vLLM`, and `SGLang`, then connect each runtime to the deployment tier where it makes the most sense.

## What you will learn

- what each major open-model runtime is optimizing for
- why the baseline `transformers` path still matters
- how local, single-node, and clustered tiers differ operationally
- how to reason from workload constraints to runtime choice

## Open-model focus only

We stay entirely in the open ecosystem. The point is not to memorize vendor products. The point is to understand the runtime shapes that repeatedly appear when teams serve open models themselves.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

In [ ]:
random.seed(2)

NOTEBOOK_ROOT = ARTIFACT_ROOT / "module_02_runtime_landscape"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def choose_label(value, thresholds):
    for boundary, label in thresholds:
        if value <= boundary:
            return label
    return thresholds[-1][1]

print("Artifact root:", NOTEBOOK_ROOT.resolve())

## Step 1: Put the main runtimes on one page

Each runtime exists because someone cared about a different bottleneck. That is why “best runtime” is the wrong question. The right question is “best runtime for which workload and which deployment tier?”

In [ ]:
runtime_df = pd.DataFrame([
    {"runtime": "transformers", "sweet_spot": "baseline and research", "strengths": "transparent Python path, easiest to inspect", "tradeoffs": "manual batching, lower serving throughput", "best_tier": "local or single-GPU baseline"},
    {"runtime": "llama.cpp", "sweet_spot": "local, CPU, edge, GGUF", "strengths": "low dependency surface, quantization breadth, laptop friendly", "tradeoffs": "not the highest GPU throughput", "best_tier": "local and edge"},
    {"runtime": "vLLM", "sweet_spot": "throughput-oriented API serving", "strengths": "continuous batching, managed KV cache, strong OpenAI-compatible serving", "tradeoffs": "heavier stack than local baselines", "best_tier": "single-node or cluster serving"},
    {"runtime": "SGLang", "sweet_spot": "advanced scheduling and structured serving", "strengths": "prefix-aware serving, strong scheduling ideas, structured-output oriented", "tradeoffs": "operational complexity is higher than raw Python baselines", "best_tier": "single-node or cluster serving"},
])
runtime_df

## Step 2: Why the `transformers` baseline still matters

You need a simple baseline before optimization means anything. The raw `transformers` path makes scheduling costs visible because it gives you very little for free.

In [ ]:
def simulate_transformers_baseline(prompt_tokens, output_tokens, concurrent_requests):
    tokenize_ms = 5.0 + 0.01 * prompt_tokens
    prefill_ms = 20.0 + 0.032 * (prompt_tokens ** 1.1)
    decode_ms = output_tokens * (2.4 + 0.0035 * prompt_tokens)
    serialized_penalty_ms = max(0, concurrent_requests - 1) * 55.0
    total_ms = tokenize_ms + prefill_ms + decode_ms + serialized_penalty_ms
    return round(total_ms, 2)

transformers_cases = []
for prompt_tokens, output_tokens in [(512, 96), (2048, 128), (4096, 256)]:
    for concurrent_requests in [1, 2, 4, 8]:
        transformers_cases.append({
            "prompt_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "concurrent_requests": concurrent_requests,
            "estimated_latency_ms": simulate_transformers_baseline(prompt_tokens, output_tokens, concurrent_requests),
        })

transformers_df = pd.DataFrame(transformers_cases)
transformers_df.head(12)

## Step 3: `llama.cpp` is the practical local baseline

`llama.cpp` is what many teams reach for when they want a transparent local runtime with GGUF models, aggressive quantization, and a tiny operational footprint. It is not just for hobby projects. It is the default answer whenever low-dependency local serving matters more than peak GPU throughput.

In [ ]:
llama_cpp_profiles = pd.DataFrame([
    {"profile": "laptop_q4", "device": "CPU or small GPU", "context_tokens": 4096, "approx_memory_gb": 5.5, "best_use": "private local prototyping"},
    {"profile": "workstation_q4", "device": "single GPU", "context_tokens": 8192, "approx_memory_gb": 10.0, "best_use": "developer workstation API"},
    {"profile": "edge_q5", "device": "small edge box", "context_tokens": 4096, "approx_memory_gb": 7.5, "best_use": "low-latency on-prem deployment"},
])
llama_cpp_profiles

## Step 4: `vLLM` exists to keep the GPU busy

`vLLM` is the reference point for high-throughput serving. The key idea is that naive request-at-a-time generation leaves throughput on the table, while continuous batching and managed KV cache keep more useful work in flight.

In [ ]:
def simulate_runtime_policy(policy, arrival_rate_rps, n_requests=240, seed=0):
    rng = random.Random(seed)
    rows = []
    arrival_clock = 0.0
    server_free_ms = 0.0
    for request_id in range(n_requests):
        arrival_clock += rng.expovariate(arrival_rate_rps) * 1000.0
        prompt_tokens = rng.choice([512, 1024, 2048, 4096])
        output_tokens = rng.choice([64, 128, 192])
        raw_service_ms = 18 + 0.025 * (prompt_tokens ** 1.08) + output_tokens * (1.9 + 0.002 * prompt_tokens)
        if policy == "fixed_batch":
            batch_window_ms = 45.0
            start_ms = max(server_free_ms, arrival_clock + batch_window_ms)
            service_ms = raw_service_ms * 0.92
        elif policy == "continuous":
            start_ms = max(server_free_ms - 18.0, arrival_clock)
            service_ms = raw_service_ms * 0.72
        else:
            start_ms = max(server_free_ms, arrival_clock)
            service_ms = raw_service_ms
        finish_ms = start_ms + service_ms
        server_free_ms = finish_ms
        rows.append({
            "policy": policy,
            "request_id": request_id,
            "wait_ms": round(start_ms - arrival_clock, 2),
            "service_ms": round(service_ms, 2),
            "latency_ms": round(finish_ms - arrival_clock, 2),
        })
    return pd.DataFrame(rows)

In [ ]:
policy_frames = []
for policy in ["sequential", "fixed_batch", "continuous"]:
    frame = simulate_runtime_policy(policy, arrival_rate_rps=2.6, n_requests=300, seed=7)
    policy_frames.append(frame)

policy_df = pd.concat(policy_frames, ignore_index=True)
policy_summary = policy_df.groupby("policy").agg(
    avg_wait_ms=("wait_ms", "mean"),
    p95_latency_ms=("latency_ms", lambda values: np.percentile(values, 95)),
    avg_service_ms=("service_ms", "mean"),
).round(2).reset_index()
policy_summary

## Step 5: `SGLang` is a scheduling and contract story too

`SGLang` is not only about raw speed. It also makes prefix sharing, structured outputs, and advanced scheduling easier to reason about. That matters when many requests resemble each other or when output shape is part of the product contract.

In [ ]:
workload_mix = pd.DataFrame([
    {"workload": "chat", "shared_prefix_ratio": 0.20, "schema_pressure": 0.20},
    {"workload": "retrieval_agent", "shared_prefix_ratio": 0.55, "schema_pressure": 0.35},
    {"workload": "tool_router", "shared_prefix_ratio": 0.45, "schema_pressure": 0.85},
    {"workload": "batch_extractor", "shared_prefix_ratio": 0.70, "schema_pressure": 0.95},
])

runtime_advantage_rows = []
for row in workload_mix.to_dict("records"):
    runtime_advantage_rows.append({
        "workload": row["workload"],
        "llama_cpp_fit": round(0.55 + 0.20 * (1 - row["schema_pressure"]), 2),
        "vllm_fit": round(0.60 + 0.35 * row["shared_prefix_ratio"], 2),
        "sglang_fit": round(0.62 + 0.25 * row["shared_prefix_ratio"] + 0.20 * row["schema_pressure"], 2),
    })

runtime_advantage_df = pd.DataFrame(runtime_advantage_rows)
runtime_advantage_df

## Step 6: Deployment tiers are about operational shape, not prestige

The same runtime can appear in different tiers, but the operational shape changes. A local tier optimizes for developer control, a single-node tier optimizes for stable serving on one machine, and a clustered tier optimizes for sustained production load.

In [ ]:
tier_df = pd.DataFrame([
    {"tier": "local", "ops_shape": "one person, one machine", "common_goal": "debugging and prototype validation", "concurrency_range": "1-5", "memory_budget_gb": 8},
    {"tier": "single_node", "ops_shape": "one service on one host", "common_goal": "small production or benchmark box", "concurrency_range": "5-50", "memory_budget_gb": 24},
    {"tier": "cluster", "ops_shape": "multiple hosts and admission control", "common_goal": "steady traffic and fault tolerance", "concurrency_range": "50+", "memory_budget_gb": 80},
])
tier_df

## Step 7: Connect workload constraints to tier choice

A deployment tier is usually forced by concurrency, context length, and required headroom. We can encode that reasoning directly.

In [ ]:
def recommend_tier(concurrency_target, context_tokens, memory_budget_gb):
    if concurrency_target <= 5 and memory_budget_gb <= 12 and context_tokens <= 8192:
        return "local"
    if concurrency_target <= 40 and memory_budget_gb <= 48 and context_tokens <= 32768:
        return "single_node"
    return "cluster"

tier_cases = []
for workload in [
    {"name": "offline analyst", "concurrency_target": 2, "context_tokens": 4096, "memory_budget_gb": 10},
    {"name": "internal API", "concurrency_target": 18, "context_tokens": 8192, "memory_budget_gb": 24},
    {"name": "public support service", "concurrency_target": 85, "context_tokens": 32768, "memory_budget_gb": 80},
]:
    tier_cases.append({**workload, "recommended_tier": recommend_tier(**{k: workload[k] for k in ["concurrency_target", "context_tokens", "memory_budget_gb"]})})

pd.DataFrame(tier_cases)

## Step 8: Build a runtime recommender

This is a deliberately simple heuristic. It encodes a few principles you can later replace with real measurements.

In [ ]:
def choose_runtime(scenario):
    if scenario["must_run_on_cpu"]:
        return "llama.cpp"
    if scenario["strict_schema"] and scenario["shared_prefix_ratio"] >= 0.4:
        return "SGLang"
    if scenario["concurrency_target"] >= 12 or scenario["throughput_priority"]:
        return "vLLM"
    return "transformers"

scenarios = pd.DataFrame([
    {"scenario": "laptop prototype", "must_run_on_cpu": True, "strict_schema": False, "shared_prefix_ratio": 0.2, "concurrency_target": 2, "throughput_priority": False},
    {"scenario": "internal coding assistant", "must_run_on_cpu": False, "strict_schema": False, "shared_prefix_ratio": 0.35, "concurrency_target": 6, "throughput_priority": False},
    {"scenario": "high-volume chat API", "must_run_on_cpu": False, "strict_schema": False, "shared_prefix_ratio": 0.55, "concurrency_target": 24, "throughput_priority": True},
    {"scenario": "JSON extraction service", "must_run_on_cpu": False, "strict_schema": True, "shared_prefix_ratio": 0.7, "concurrency_target": 14, "throughput_priority": True},
])
scenarios["recommended_runtime"] = scenarios.apply(lambda row: choose_runtime(row.to_dict()), axis=1)
scenarios

In [ ]:
runtime_counts = scenarios["recommended_runtime"].value_counts().rename_axis("runtime").reset_index(name="count")
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(runtime_counts["runtime"], runtime_counts["count"], color=["#4c78a8", "#f58518", "#54a24b", "#e45756"])
ax.set_title("Heuristic runtime recommendations across sample scenarios")
ax.set_ylabel("scenario count")
plt.show()

## Step 9: Contracts should travel across runtimes

Even when the serving engine changes, application contracts should stay stable. OpenAI-compatible request and response shapes are popular because they let the application layer survive runtime swaps.

In [ ]:
request_examples = [
    {"model": "open-model-a", "messages": [{"role": "user", "content": "Summarize the incident."}], "temperature": 0.2},
    {"model": "open-model-b", "messages": [{"role": "system", "content": "Return JSON."}, {"role": "user", "content": "Extract fields."}], "temperature": 0.0},
]

def validate_request_shape(payload):
    required = {"model", "messages"}
    missing = sorted(required - set(payload))
    return {
        "valid": not missing and isinstance(payload["messages"], list),
        "missing": missing,
        "message_count": len(payload.get("messages", [])) if isinstance(payload.get("messages"), list) else 0,
    }

validation_rows = [validate_request_shape(payload) | {"model": payload["model"]} for payload in request_examples]
pd.DataFrame(validation_rows)

## Step 10: Plan migration between tiers

Most teams should not jump from a notebook straight to a cluster. A better path is baseline first, then a benchmarked single-node service, then scale out only after the workload justifies it.

In [ ]:
roadmap = pd.DataFrame([
    {"stage": 1, "move": "Notebook baseline", "runtime": "transformers or llama.cpp", "proof_needed": "quality and basic latency sanity"},
    {"stage": 2, "move": "Local API prototype", "runtime": "llama.cpp or transformers", "proof_needed": "contract stability and repeatable prompts"},
    {"stage": 3, "move": "Single-node benchmark box", "runtime": "vLLM or SGLang", "proof_needed": "p95 latency, throughput, memory headroom"},
    {"stage": 4, "move": "Scaled serving", "runtime": "vLLM or SGLang", "proof_needed": "admission control, rollback, observability"},
])
roadmap

## Step 11: Diagnose the bottleneck before choosing the cure

Runtime choice is downstream of bottleneck diagnosis. If the pain is CPU locality, choose differently than if the pain is queue growth or schema enforcement.

In [ ]:
diagnostic_cases = pd.DataFrame([
    {"case": "developer laptop", "ttft_ms": 420, "queue_ms": 12, "schema_fail_rate": 0.02, "cpu_only": 1},
    {"case": "busy internal API", "ttft_ms": 960, "queue_ms": 320, "schema_fail_rate": 0.01, "cpu_only": 0},
    {"case": "JSON extraction service", "ttft_ms": 740, "queue_ms": 110, "schema_fail_rate": 0.14, "cpu_only": 0},
])

def classify_bottleneck(row):
    if row["cpu_only"]:
        return "hardware locality"
    if row["schema_fail_rate"] >= 0.1:
        return "contract enforcement"
    if row["queue_ms"] >= 200:
        return "throughput pressure"
    return "general latency"

diagnostic_cases["primary_bottleneck"] = diagnostic_cases.apply(classify_bottleneck, axis=1)
diagnostic_cases

## Step 12: Export a runtime recommendation artifact

We will save a small artifact that records the heuristic recommendations for the sample scenarios. Later notebooks can reuse the same structure with real benchmark numbers.

In [ ]:
runtime_report = {
    "generated_by": "notebook_02_runtime_landscape",
    "scenarios": scenarios.to_dict("records"),
    "policy_summary": policy_summary.to_dict("records"),
}
report_path = NOTEBOOK_ROOT / "runtime_recommendations.json"
with report_path.open("w", encoding="utf-8") as handle:
    json.dump(runtime_report, handle, indent=2)
print("Saved:", report_path.resolve())

## Exercises

1. Add a scenario where memory is plentiful but schema pressure is low. Does the recommender still pick the runtime you expect?
2. Change the arrival rate in the policy simulator and compare `fixed_batch` versus `continuous` again.
3. Replace one of the tier thresholds with your own organization's operational constraints.

In [ ]:
scenario_template = pd.DataFrame([
    {"scenario": "your_service", "must_run_on_cpu": False, "strict_schema": False, "shared_prefix_ratio": 0.0, "concurrency_target": 1, "throughput_priority": False},
])
scenario_template

## Recap

You now have a mental map of the main open-model runtimes and the deployment tiers they inhabit. In the next notebook we will zoom into the request itself: prefill, decode, batch effects, and memory budgets.

In [ ]:
assert policy_summary["p95_latency_ms"].max() > policy_summary["p95_latency_ms"].min()
assert scenarios["recommended_runtime"].isin(["transformers", "llama.cpp", "vLLM", "SGLang"]).all()
assert tier_df["tier"].tolist() == ["local", "single_node", "cluster"]
print("Notebook 02 sanity checks passed.")